# TinyAudioTransformer: Phase 0–1 (correctness before training)

**Notebook settings:** Accelerator = **GPU T4 x2**, Internet = **On** (downloads Speech Commands v2, ~2.3 GB, and the AST reference weights, ~350 MB).

What this notebook does:
1. **Phase 0, ground truth:** downloads Speech Commands v2 (35 words), checks the official split is speaker-disjoint, and caches Kaldi fbank features.
2. **Phase 1, correctness:** runs the test suite, including **parity with the authors' AST**. Our modules loaded with their weights must reproduce their logits.
3. **Learnability:** overfits one real batch.
4. **Sanity run:** trains TinyAudioTransformer (GPU 0) and a CNN baseline (GPU 1) in parallel, selects on val, and evaluates on test once.

Generated by `kaggle/build_notebook.py`. Edit the repo files, not these cells.

In [ ]:
import os, sys, subprocess, time, json, pathlib
os.makedirs("/kaggle/working/tat/tests", exist_ok=True)
os.chdir("/kaggle/working/tat")
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
try:
    import transformers; print("transformers", transformers.__version__)
except ImportError:
    print("transformers missing → AST parity tests will be skipped")
!pip install -q pytest

## Source files

In [ ]:
%%writefile /kaggle/working/tat/features.py
# features.py
# Kaldi-compatible log-mel filterbank in plain torch, matching
# torchaudio.compliance.kaldi.fbank(wav, sample_frequency=16000, window_type="hanning", num_mel_bins=128)
# which is what AST (and HF's ASTFeatureExtractor) use. Owning it removes the torchaudio dependency,
# works on batches/GPU, and gives one reference to port when the feature pipeline moves to the browser.
# tests/test_features.py checks it against torchaudio and the HF extractor.
import math

import torch
import torch.nn.functional as F

SAMPLE_RATE = 16000
FRAME_LENGTH = 400      # 25 ms
FRAME_SHIFT = 160       # 10 ms
N_FFT = 512             # frame length rounded up to a power of two
PREEMPHASIS = 0.97
LOW_FREQ = 20.0
EPS = torch.finfo(torch.float32).eps


def _mel(f):
    return 1127.0 * torch.log1p(f / 700.0)


def mel_banks(n_mels=128, sample_rate=SAMPLE_RATE, n_fft=N_FFT, low_freq=LOW_FREQ):
    # Triangles evenly spaced on the mel scale, evaluated at FFT bin centres; Nyquist bin gets weight 0
    mel_lo = 1127.0 * math.log1p(low_freq / 700.0)
    mel_hi = 1127.0 * math.log1p((sample_rate / 2) / 700.0)
    delta = (mel_hi - mel_lo) / (n_mels + 1)

    b = torch.arange(n_mels).unsqueeze(1)
    left, center, right = (mel_lo + (b + k) * delta for k in (0.0, 1.0, 2.0))
    mel = _mel(sample_rate / n_fft * torch.arange(n_fft // 2)).unsqueeze(0)

    up = (mel - left) / (center - left)
    down = (right - mel) / (right - center)
    banks = torch.clamp(torch.min(up, down), min=0.0)        # [n_mels, n_fft/2]
    return F.pad(banks, (0, 1))                               # [n_mels, n_fft/2 + 1]


def fbank(wav, n_mels=128, n_frames=None):
    """wav: float waveform in [-1, 1], shape [N] or [B, N] at 16 kHz.
    Returns log-mel [T, n_mels] (or [B, T, n_mels]); if n_frames is given, zero-pads/crops time to it."""
    squeeze = wav.dim() == 1
    wav = wav.reshape(-1, wav.shape[-1]).float()

    frames = wav.unfold(-1, FRAME_LENGTH, FRAME_SHIFT)                         # [B, T, 400], snip_edges=True
    frames = frames - frames.mean(dim=-1, keepdim=True)                         # remove DC per frame
    prev = torch.cat((frames[..., :1], frames[..., :-1]), dim=-1)
    frames = frames - PREEMPHASIS * prev                                        # pre-emphasis
    frames = frames * torch.hann_window(FRAME_LENGTH, periodic=False, device=wav.device)
    frames = F.pad(frames, (0, N_FFT - FRAME_LENGTH))

    power = torch.fft.rfft(frames).abs().pow(2)                                 # [B, T, 257]
    mel = power @ mel_banks(n_mels).to(wav.device).T                            # [B, T, n_mels]
    out = torch.clamp(mel, min=EPS).log()

    if n_frames is not None:
        out = pad_or_crop(out, n_frames)
    return out[0] if squeeze else out


def pad_or_crop(x, n_frames):
    # x: [..., T, F]; zero-pad at the end of time like AST's ZeroPad2d, or crop
    T = x.shape[-2]
    if T < n_frames:
        return F.pad(x, (0, 0, 0, n_frames - T))
    return x[..., :n_frames, :]

In [ ]:
%%writefile /kaggle/working/tat/patch_embed.py
# patch_embed.py
import torch
import torch.nn as nn


def num_patches_1d(size, patch, stride):
    # Number of conv windows along one axis (no padding), same formula as AST's get_shape
    return (size - patch) // stride + 1


class PatchEmbed(nn.Module):
    """Spectrogram -> sequence of patch embeddings.

    Input follows the AST convention: [B, T, F] (time frames, mel bins), which is also
    what kaldi fbank returns. With stride < patch the patches overlap (AST: 16x16, stride 10).
    """

    def __init__(self, n_mels=128, n_frames=128, patch_size=16, patch_stride=10, embed_dim=128):
        super().__init__()
        self.n_mels = n_mels
        self.n_frames = n_frames
        self.grid = (num_patches_1d(n_mels, patch_size, patch_stride),
                     num_patches_1d(n_frames, patch_size, patch_stride))
        self.num_patches = self.grid[0] * self.grid[1]

        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_stride)

    def forward(self, x):
        # x: [B, T, F]
        if x.shape[1:] != (self.n_frames, self.n_mels):
            raise ValueError(f"expected input [B, {self.n_frames}, {self.n_mels}], got {list(x.shape)}")
        x = x.unsqueeze(1).transpose(2, 3)  # → [B, 1, F, T]
        x = self.proj(x)                    # → [B, embed_dim, F', T']
        x = x.flatten(2).transpose(1, 2)    # → [B, F'*T', embed_dim]  (frequency-major order, as in AST)
        return x

In [ ]:
%%writefile /kaggle/working/tat/transformer_block.py
# transformer_block.py
import torch
import torch.nn as nn


class TransformerEncoderBlock(nn.Module):
    """Pre-norm ViT block: x + Attn(LN(x)), then x + MLP(LN(x))."""

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1, attn_dropout=0.0, ln_eps=1e-6):
        super().__init__()
        hidden = int(embed_dim * mlp_ratio)

        self.norm1 = nn.LayerNorm(embed_dim, eps=ln_eps)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=attn_dropout, batch_first=True)
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim, eps=ln_eps)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)  # normalise once, use as q, k and v
        x = x + self.drop1(self.attn(h, h, h, need_weights=False)[0])
        x = x + self.mlp(self.norm2(x))
        return x

In [ ]:
%%writefile /kaggle/working/tat/model.py
# model.py
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn

from patch_embed import PatchEmbed
from transformer_block import TransformerEncoderBlock


@dataclass
class TATConfig:
    n_mels: int = 128
    n_frames: int = 128          # 1 s clip = 98 kaldi frames, zero-padded to 128 (AST speech-commands recipe)
    patch_size: int = 16
    patch_stride: int = 10       # < patch_size → overlapping patches (AST); set to 16 for ViT-style
    embed_dim: int = 128
    depth: int = 4
    num_heads: int = 4
    mlp_ratio: float = 4.0
    num_classes: int = 35
    dropout: float = 0.1
    attn_dropout: float = 0.0
    num_prefix_tokens: int = 1   # 1 = [CLS]; 2 = [CLS]+[DIST] (AST/DeiT), pooled as their mean
    head_norm: bool = False      # extra LayerNorm inside the head (AST's ASTMLPHead has one)
    ln_eps: float = 1e-6


class FeatureNorm(nn.Module):
    """(x - mean) / (2 * std), as in AST. Stats live in the model so they ship with the checkpoint."""

    def __init__(self, mean=0.0, std=0.5):
        super().__init__()
        self.register_buffer("mean", torch.tensor(float(mean)))
        self.register_buffer("std", torch.tensor(float(std)))

    def set_stats(self, mean, std):
        self.mean.fill_(float(mean))
        self.std.fill_(float(std))

    def forward(self, x):
        return (x - self.mean) / (2 * self.std)


class TinyAudioTransformer(nn.Module):
    def __init__(self, cfg: TATConfig = TATConfig()):
        super().__init__()
        self.cfg = cfg
        self.norm_in = FeatureNorm()

        self.patch_embed = PatchEmbed(cfg.n_mels, cfg.n_frames, cfg.patch_size, cfg.patch_stride, cfg.embed_dim)
        n_tokens = self.patch_embed.num_patches + cfg.num_prefix_tokens

        self.prefix_tokens = nn.Parameter(torch.zeros(1, cfg.num_prefix_tokens, cfg.embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens, cfg.embed_dim))
        self.pos_drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.Sequential(*[
            TransformerEncoderBlock(cfg.embed_dim, cfg.num_heads, cfg.mlp_ratio,
                                    cfg.dropout, cfg.attn_dropout, cfg.ln_eps)
            for _ in range(cfg.depth)
        ])
        self.norm = nn.LayerNorm(cfg.embed_dim, eps=cfg.ln_eps)
        self.head_norm = nn.LayerNorm(cfg.embed_dim, eps=cfg.ln_eps) if cfg.head_norm else nn.Identity()
        self.head = nn.Linear(cfg.embed_dim, cfg.num_classes)

        self.apply(self._init_weights)
        nn.init.trunc_normal_(self.prefix_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    @staticmethod
    def _init_weights(m):
        # ViT/DeiT init: small truncated normal for linears, identity LayerNorms
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.MultiheadAttention):
            nn.init.trunc_normal_(m.in_proj_weight, std=0.02)
            nn.init.zeros_(m.in_proj_bias)

    def forward(self, x):
        # x: raw log-mel fbank [B, T, F]
        B = x.shape[0]
        x = self.norm_in(x)
        x = self.patch_embed(x)                                 # → [B, N, D]

        prefix = self.prefix_tokens.expand(B, -1, -1)           # → [B, P, D]
        x = torch.cat((prefix, x), dim=1) + self.pos_embed      # → [B, P+N, D]
        x = self.pos_drop(x)

        x = self.blocks(x)
        x = self.norm(x)

        pooled = x[:, :self.cfg.num_prefix_tokens].mean(dim=1)  # [CLS] (or mean of [CLS],[DIST])
        return self.head(self.head_norm(pooled))

    def config_dict(self):
        return asdict(self.cfg)

In [ ]:
%%writefile /kaggle/working/tat/baselines.py
# baselines.py
# A small CNN on the same features. If the transformer cannot beat this, the problem is the transformer
# (or its training), not the data pipeline.
from dataclasses import dataclass, asdict

import torch.nn as nn

from model import FeatureNorm


@dataclass
class CNNConfig:
    channels: tuple = (32, 64, 128, 128)
    num_classes: int = 35
    dropout: float = 0.1


class CNNBaseline(nn.Module):
    def __init__(self, cfg: CNNConfig = CNNConfig()):
        super().__init__()
        self.cfg = cfg
        self.norm_in = FeatureNorm()
        layers, c_in = [], 1
        for c in cfg.channels:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(cfg.dropout),
                                  nn.Linear(c_in, cfg.num_classes))

    def forward(self, x):
        # x: raw log-mel fbank [B, T, F]
        x = self.norm_in(x).unsqueeze(1)  # → [B, 1, T, F]
        return self.head(self.features(x))

    def config_dict(self):
        return asdict(self.cfg)

In [ ]:
%%writefile /kaggle/working/tat/dataset.py
# dataset.py
# Speech Commands v2, 35 words, official split (validation_list.txt / testing_list.txt).
# The official lists are speaker-disjoint: the part of the filename before "_nohash_" is the speaker id.
import tarfile
import urllib.request
import wave
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset

from features import SAMPLE_RATE, fbank

SC_V2_URL = "http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz"
EXPECTED_SIZES = {"train": 84843, "val": 9981, "test": 11005}


def download_speech_commands(root):
    root = Path(root)
    if (root / "testing_list.txt").exists():
        return root
    root.mkdir(parents=True, exist_ok=True)
    archive = root / "speech_commands_v0.02.tar.gz"
    if not archive.exists():
        print(f"Downloading {SC_V2_URL} (~2.3 GB)")
        urllib.request.urlretrieve(SC_V2_URL, archive)
    print("Extracting")
    with tarfile.open(archive) as tar:
        kwargs = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
        tar.extractall(root, **kwargs)
    archive.unlink()
    return root


def speaker_id(rel_path):
    return Path(rel_path).name.split("_nohash_")[0]


def class_names(root):
    return sorted(d.name for d in Path(root).iterdir() if d.is_dir() and not d.name.startswith("_"))


def official_splits(root):
    """Returns {split: [(relative_path, label_idx), ...]}, classes."""
    root = Path(root)
    classes = class_names(root)
    idx = {c: i for i, c in enumerate(classes)}
    val = set(root.joinpath("validation_list.txt").read_text().split())
    test = set(root.joinpath("testing_list.txt").read_text().split())

    splits = {"train": [], "val": [], "test": []}
    for c in classes:
        for f in sorted((root / c).glob("*.wav")):
            rel = f"{c}/{f.name}"
            split = "val" if rel in val else "test" if rel in test else "train"
            splits[split].append((rel, idx[c]))
    return splits, classes


def load_wav(path):
    # Speech Commands is 16-bit mono PCM; scale to [-1, 1] like torchaudio.load
    with wave.open(str(path), "rb") as w:
        assert w.getframerate() == SAMPLE_RATE and w.getnchannels() == 1 and w.getsampwidth() == 2, path
        pcm = np.frombuffer(w.readframes(w.getnframes()), dtype="<i2")
    return torch.from_numpy(pcm.astype(np.float32) / 32768.0)


class SpeechCommandsFbank(Dataset):
    def __init__(self, root, items, n_mels=128, n_frames=128):
        self.root, self.items = Path(root), items
        self.n_mels, self.n_frames = n_mels, n_frames

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        rel, label = self.items[i]
        return fbank(load_wav(self.root / rel), self.n_mels, self.n_frames), label


def build_cache(root, cache_dir, n_mels=128, n_frames=128, num_workers=4):
    """Computes features once per split and stores them as fp16 tensors.
    Train mean/std are computed on the padded train features only (the model sees padding too)."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    splits, classes = official_splits(root)

    for split, items in splits.items():
        out = cache_dir / f"{split}.pt"
        if out.exists():
            continue
        loader = DataLoader(SpeechCommandsFbank(root, items, n_mels, n_frames),
                            batch_size=512, num_workers=num_workers)
        x = torch.empty(len(items), n_frames, n_mels, dtype=torch.float16)
        y = torch.empty(len(items), dtype=torch.long)
        n = 0
        s, s2 = 0.0, 0.0
        for feats, labels in loader:
            x[n:n + len(feats)] = feats.half()
            y[n:n + len(feats)] = labels
            s += feats.double().sum().item()
            s2 += feats.double().pow(2).sum().item()
            n += len(feats)
        count = x.numel()
        mean = s / count
        std = (s2 / count - mean ** 2) ** 0.5
        torch.save({"x": x, "y": y, "files": [r for r, _ in items], "classes": classes,
                    "mean": mean, "std": std}, out)
        print(f"{split}: {tuple(x.shape)}  mean={mean:.4f} std={std:.4f}")


def load_cache(cache_dir, split):
    return torch.load(Path(cache_dir) / f"{split}.pt", weights_only=False)

In [ ]:
%%writefile /kaggle/working/tat/prepare_data.py
# prepare_data.py
# Phase 0: download Speech Commands v2, check the official split, cache fbank features.
import argparse
from collections import Counter

from dataset import EXPECTED_SIZES, build_cache, download_speech_commands, official_splits, speaker_id


def check_splits(root):
    splits, classes = official_splits(root)
    assert len(classes) == 35, f"expected 35 words, found {len(classes)}: {classes}"

    speakers = {s: {speaker_id(r) for r, _ in items} for s, items in splits.items()}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        shared = speakers[a] & speakers[b]
        assert not shared, f"{len(shared)} speakers in both {a} and {b}"

    for s, items in splits.items():
        per_class = Counter(label for _, label in items)
        print(f"{s:>5}: {len(items):6d} clips  {len(speakers[s]):5d} speakers  "
              f"{len(per_class)} classes  (min/max per class {min(per_class.values())}/{max(per_class.values())})")
        if len(items) != EXPECTED_SIZES[s]:
            print(f"  WARNING: expected {EXPECTED_SIZES[s]} clips for {s}")
    print("Splits are speaker-disjoint.")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="data/speech_commands_v2")
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--num-workers", type=int, default=4)
    args = p.parse_args()

    download_speech_commands(args.root)
    check_splits(args.root)
    build_cache(args.root, args.cache_dir, num_workers=args.num_workers)

In [ ]:
%%writefile /kaggle/working/tat/ast_reference.py
# ast_reference.py
# Loads the authors' fine-tuned AST (MIT/ast-finetuned-speech-commands-v2, via HF transformers) into
# TinyAudioTransformer. If our modules compute the same function as AST, logits match to float precision;
# any architectural misunderstanding (patch order, token pooling, LayerNorm placement, attention layout)
# shows up as a mismatch. Used by tests/test_parity_ast.py.
import torch

from model import TATConfig, TinyAudioTransformer

HF_AST_SC = "MIT/ast-finetuned-speech-commands-v2"


def config_from_hf(hf_cfg):
    assert hf_cfg.frequency_stride == hf_cfg.time_stride, "TAT uses one stride for both axes"
    return TATConfig(
        n_mels=hf_cfg.num_mel_bins, n_frames=hf_cfg.max_length,
        patch_size=hf_cfg.patch_size, patch_stride=hf_cfg.time_stride,
        embed_dim=hf_cfg.hidden_size, depth=hf_cfg.num_hidden_layers, num_heads=hf_cfg.num_attention_heads,
        mlp_ratio=hf_cfg.intermediate_size / hf_cfg.hidden_size, num_classes=hf_cfg.num_labels,
        dropout=0.0, num_prefix_tokens=2, head_norm=True, ln_eps=hf_cfg.layer_norm_eps,
    )


# transformers renamed AST's modules over versions; rewrite older spellings to the 5.x ones.
# Order matters: attention.output.dense must be renamed before the MLP's output.dense.
_RENAMES = [
    (".encoder.layer.", ".layers."),
    (".attention.attention.query.", ".attention.q_proj."),
    (".attention.attention.key.", ".attention.k_proj."),
    (".attention.attention.value.", ".attention.v_proj."),
    (".attention.output.dense.", ".attention.o_proj."),
    (".intermediate.dense.", ".mlp.fc1."),
    (".output.dense.", ".mlp.fc2."),
]


class _TrackedDict(dict):
    # Records which tensors were read, so a mapping that silently skips weights fails loudly
    def __init__(self, *a):
        super().__init__(*a)
        self.used = set()

    def __getitem__(self, k):
        self.used.add(k)
        return super().__getitem__(k)


def _to_current_names(sd):
    out = _TrackedDict()
    for k, v in sd.items():
        for old, new in _RENAMES:
            k = k.replace(old, new)
        out[k] = v
    return out


def map_hf_state_dict(sd, depth):
    sd = _to_current_names(sd)
    e = "audio_spectrogram_transformer.embeddings."
    out = {
        "prefix_tokens": torch.cat([sd[e + "cls_token"], sd[e + "distillation_token"]], dim=1),
        "pos_embed": sd[e + "position_embeddings"],
        "patch_embed.proj.weight": sd[e + "patch_embeddings.projection.weight"],
        "patch_embed.proj.bias": sd[e + "patch_embeddings.projection.bias"],
        "norm.weight": sd["audio_spectrogram_transformer.layernorm.weight"],
        "norm.bias": sd["audio_spectrogram_transformer.layernorm.bias"],
        "head_norm.weight": sd["classifier.layernorm.weight"],
        "head_norm.bias": sd["classifier.layernorm.bias"],
        "head.weight": sd["classifier.dense.weight"],
        "head.bias": sd["classifier.dense.bias"],
    }
    for i in range(depth):
        h, o = f"audio_spectrogram_transformer.layers.{i}.", f"blocks.{i}."
        for p in ("weight", "bias"):
            qkv = [sd[h + f"attention.{n}_proj.{p}"] for n in "qkv"]
            out[o + f"attn.in_proj_{p}"] = torch.cat(qkv, dim=0)
            out[o + f"attn.out_proj.{p}"] = sd[h + f"attention.o_proj.{p}"]
            out[o + f"norm1.{p}"] = sd[h + f"layernorm_before.{p}"]
            out[o + f"norm2.{p}"] = sd[h + f"layernorm_after.{p}"]
            out[o + f"mlp.0.{p}"] = sd[h + f"mlp.fc1.{p}"]
            out[o + f"mlp.3.{p}"] = sd[h + f"mlp.fc2.{p}"]
    unused = set(sd) - sd.used
    assert not unused, f"AST tensors not mapped: {sorted(unused)[:5]}"
    return out


def load_ast_into_tat(name=HF_AST_SC):
    """Returns (our model with AST weights, the HF model, the HF feature extractor)."""
    from transformers import ASTFeatureExtractor, ASTForAudioClassification

    hf = ASTForAudioClassification.from_pretrained(name).eval()
    fe = ASTFeatureExtractor.from_pretrained(name)
    cfg = config_from_hf(hf.config)

    ours = TinyAudioTransformer(cfg).eval()
    missing, unexpected = ours.load_state_dict(map_hf_state_dict(hf.state_dict(), cfg.depth), strict=False)
    assert not unexpected, unexpected
    assert set(missing) == {"norm_in.mean", "norm_in.std"}, missing
    ours.norm_in.set_stats(fe.mean, fe.std)
    return ours, hf, fe

In [ ]:
%%writefile /kaggle/working/tat/train.py
# train.py
# Plain supervised training on cached features (no augmentation yet; that is Phase 2).
# Picks the checkpoint on val, then evaluates on test exactly once.
#   python train.py --model tat --cache-dir data/cache --out-dir runs/tat_s0
#   python train.py --model tat --overfit-batch 64      # sanity check: must reach 100% on one batch
import argparse
import json
import math
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from baselines import CNNBaseline, CNNConfig
from dataset import load_cache
from model import TATConfig, TinyAudioTransformer


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--model", choices=["tat", "cnn"], default="tat")
    p.add_argument("--cache-dir", default="data/cache")
    p.add_argument("--out-dir", default="runs/tat")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--weight-decay", type=float, default=0.05)
    p.add_argument("--warmup-epochs", type=float, default=2)
    p.add_argument("--label-smoothing", type=float, default=0.0)
    p.add_argument("--grad-clip", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--no-amp", action="store_true")
    p.add_argument("--limit", type=int, default=0, help="use only N clips per split (smoke tests)")
    p.add_argument("--overfit-batch", type=int, default=0, help="train on one batch of N clips until 100%%")
    # TinyAudioTransformer size
    p.add_argument("--embed-dim", type=int, default=128)
    p.add_argument("--depth", type=int, default=4)
    p.add_argument("--heads", type=int, default=4)
    p.add_argument("--mlp-ratio", type=float, default=4.0)
    p.add_argument("--patch-stride", type=int, default=10)
    p.add_argument("--dropout", type=float, default=0.1)
    return p.parse_args(argv)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def build_model(args, num_classes):
    if args.model == "cnn":
        return CNNBaseline(CNNConfig(num_classes=num_classes))
    return TinyAudioTransformer(TATConfig(
        embed_dim=args.embed_dim, depth=args.depth, num_heads=args.heads, mlp_ratio=args.mlp_ratio,
        patch_stride=args.patch_stride, dropout=args.dropout, num_classes=num_classes))


def load_split(cache_dir, split, device, limit=0):
    d = load_cache(cache_dir, split)
    if limit:
        idx = torch.randperm(len(d["y"]), generator=torch.Generator().manual_seed(0))[:limit]
        d["x"], d["y"] = d["x"][idx], d["y"][idx]
        d["files"] = [d["files"][i] for i in idx.tolist()]
    # Whole split fits on a T4 in fp16 (train ≈ 2.8 GB), so batching is just indexing
    d["x"], d["y"] = d["x"].to(device), d["y"].to(device)
    return d


@torch.no_grad()
def predict(model, x, batch_size, amp):
    model.eval()
    out = []
    for i in range(0, len(x), batch_size):
        with torch.autocast(x.device.type, dtype=torch.float16, enabled=amp):
            out.append(model(x[i:i + batch_size].float()).float())
    return torch.cat(out)


def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item() * 100


def overfit_one_batch(model, train, args, device, amp, max_steps=1000):
    g = torch.Generator().manual_seed(args.seed)
    idx = torch.randperm(len(train["y"]), generator=g)[:args.overfit_batch].to(device)
    x, y = train["x"][idx].float(), train["y"][idx]
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.0)
    for step in range(1, max_steps + 1):
        model.train()
        loss = nn.functional.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % 10 == 0:
            acc = accuracy(predict(model, x, len(x), amp), y)
            print(f"step {step:4d}  loss {loss.item():.4f}  acc {acc:.1f}%", flush=True)
            if acc == 100.0:
                print(f"PASS: memorised {len(y)} clips in {step} steps")
                return True
    print(f"FAIL: did not reach 100% on {len(y)} clips in {max_steps} steps")
    return False


def main(argv=None):
    args = parse_args(argv)
    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda" and not args.no_amp
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    train = load_split(args.cache_dir, "train", device, args.limit)
    classes = train["classes"]
    model = build_model(args, len(classes)).to(device)
    model.norm_in.set_stats(train["mean"], train["std"])  # train-split stats only
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{args.model}: {n_params / 1e6:.3f}M params, {len(train['y'])} train clips, device {device}")

    if args.overfit_batch:
        sys.exit(0 if overfit_one_batch(model, train, args, device, amp) else 1)

    val = load_split(args.cache_dir, "val", device, args.limit)

    # No weight decay on biases, norms, tokens and position embeddings
    decay = [p for n, p in model.named_parameters() if p.ndim >= 2 and "pos_embed" not in n and "prefix" not in n]
    no_decay = [p for n, p in model.named_parameters() if not (p.ndim >= 2 and "pos_embed" not in n and "prefix" not in n)]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": args.weight_decay},
                             {"params": no_decay, "weight_decay": 0.0}], lr=args.lr)

    steps_per_epoch = math.ceil(len(train["y"]) / args.batch_size)
    total, warmup = args.epochs * steps_per_epoch, int(args.warmup_epochs * steps_per_epoch)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / max(1, warmup) if s < warmup
                                              else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    criterion = nn.CrossEntropyLoss(label_smoothing=args.label_smoothing)
    g = torch.Generator(device=device).manual_seed(args.seed)

    history, best_val, best_epoch = [], -1.0, -1
    t0 = time.time()
    for epoch in range(1, args.epochs + 1):
        model.train()
        perm = torch.randperm(len(train["y"]), device=device, generator=g)
        loss_sum, correct, seen = 0.0, 0, 0
        for i in range(0, len(perm), args.batch_size):
            idx = perm[i:i + args.batch_size]
            x, y = train["x"][idx].float(), train["y"][idx]
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                logits = model(x)
                loss = criterion(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(opt)
            scaler.update()
            sched.step()

            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            seen += len(y)

        val_acc = accuracy(predict(model, val["x"], 1024, amp), val["y"])
        history.append({"epoch": epoch, "train_loss": loss_sum / seen, "train_acc": 100 * correct / seen,
                        "val_acc": val_acc, "lr": sched.get_last_lr()[0], "time_s": time.time() - t0})
        print(f"epoch {epoch:3d}  loss {loss_sum / seen:.4f}  train {100 * correct / seen:5.2f}%  "
              f"val {val_acc:5.2f}%  lr {sched.get_last_lr()[0]:.2e}  {time.time() - t0:.0f}s", flush=True)
        if val_acc > best_val:
            best_val, best_epoch = val_acc, epoch
            torch.save({"model": args.model, "config": model.config_dict(), "state_dict": model.state_dict(),
                        "classes": classes, "epoch": epoch, "val_acc": val_acc}, out_dir / "best.pt")

    # Test once, with the val-selected checkpoint
    del train
    model.load_state_dict(torch.load(out_dir / "best.pt", map_location=device)["state_dict"])
    test = load_split(args.cache_dir, "test", device, args.limit)
    logits = predict(model, test["x"], 1024, amp)
    preds, y = logits.argmax(1).cpu(), test["y"].cpu()
    cm = torch.zeros(len(classes), len(classes), dtype=torch.long)
    cm.index_put_((y, preds), torch.ones_like(y), accumulate=True)
    per_class = (cm.diag().float() / cm.sum(1).clamp(min=1) * 100).tolist()

    results = {
        "model": args.model, "config": model.config_dict(), "params": n_params, "seed": args.seed,
        "best_epoch": best_epoch, "best_val_acc": best_val, "test_acc": accuracy(logits, test["y"]),
        "per_class_test_acc": dict(zip(classes, per_class)), "confusion_matrix": cm.tolist(),
        "train_time_s": time.time() - t0, "args": vars(args),
    }
    (out_dir / "results.json").write_text(json.dumps(results, indent=2))
    (out_dir / "history.json").write_text(json.dumps(history, indent=2))
    torch.save({"logits": logits.cpu(), "y": y, "files": test["files"], "classes": classes}, out_dir / "test_logits.pt")
    print(f"\nbest val {best_val:.2f}% (epoch {best_epoch})  →  TEST {results['test_acc']:.2f}%")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/tat/tests/conftest.py
import os
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))


@pytest.fixture(scope="session")
def sc_root():
    # Set SC_ROOT to the extracted Speech Commands v2 folder to run data tests
    root = os.environ.get("SC_ROOT")
    if not root or not Path(root, "testing_list.txt").exists():
        pytest.skip("SC_ROOT not set to an extracted Speech Commands v2 folder")
    return Path(root)


def hf_or_skip():
    # Parity tests need transformers + network access to the Hugging Face Hub (or a warm cache)
    pytest.importorskip("transformers")
    try:
        from ast_reference import load_ast_into_tat
        return load_ast_into_tat()
    except OSError as e:
        pytest.skip(f"cannot load AST from the Hub: {e}")

In [ ]:
%%writefile /kaggle/working/tat/tests/test_features.py
import math

import pytest
import torch

from features import SAMPLE_RATE, fbank


def _test_signal(seconds=1.0, seed=0):
    # Chirp + noise: exercises the whole spectrum, unlike pure noise
    g = torch.Generator().manual_seed(seed)
    t = torch.arange(int(seconds * SAMPLE_RATE)) / SAMPLE_RATE
    chirp = 0.3 * torch.sin(2 * math.pi * (100 + 3000 * t) * t)
    return chirp + 0.05 * torch.randn(len(t), generator=g)


@pytest.mark.parametrize("seconds,real_frames", [(0.3, 28), (1.0, 98), (1.5, 148)])
def test_fixed_output_shape(seconds, real_frames):
    out = fbank(_test_signal(seconds), n_mels=128, n_frames=128)
    assert out.shape == (128, 128)
    if real_frames < 128:
        assert torch.all(out[real_frames:] == 0), "padding must be zeros, as in AST"
        assert torch.all(out[:real_frames] != 0)


def test_batch_matches_single():
    wavs = torch.stack([_test_signal(seed=s) for s in range(3)])
    batched = fbank(wavs, n_frames=128)
    for i in range(3):
        torch.testing.assert_close(batched[i], fbank(wavs[i], n_frames=128))


def test_matches_torchaudio_kaldi():
    kaldi = pytest.importorskip("torchaudio.compliance.kaldi")
    wav = _test_signal()
    ref = kaldi.fbank(wav.unsqueeze(0), sample_frequency=SAMPLE_RATE, window_type="hanning", num_mel_bins=128)
    torch.testing.assert_close(fbank(wav), ref, atol=1e-3, rtol=1e-4)


def test_matches_hf_ast_feature_extractor():
    transformers = pytest.importorskip("transformers")
    try:
        fe = transformers.ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-speech-commands-v2")
    except OSError as e:
        pytest.skip(f"cannot reach the Hub: {e}")
    wav = _test_signal()
    hf = fe(wav.numpy(), sampling_rate=SAMPLE_RATE, return_tensors="pt").input_values[0]
    hf_raw = hf * (2 * fe.std) + fe.mean  # undo AST normalisation; ours lives inside the model
    torch.testing.assert_close(fbank(wav, 128, fe.max_length), hf_raw, atol=1e-3, rtol=1e-4)

In [ ]:
%%writefile /kaggle/working/tat/tests/test_model.py
import pytest
import torch
import torch.nn as nn

from baselines import CNNBaseline, CNNConfig
from model import TATConfig, TinyAudioTransformer


@pytest.mark.parametrize("stride,grid", [(10, (12, 12)), (16, (8, 8))])
def test_patch_count_matches_conv_output(stride, grid):
    m = TinyAudioTransformer(TATConfig(patch_stride=stride))
    assert m.patch_embed.grid == grid
    with torch.no_grad():
        conv_out = m.patch_embed.proj(torch.zeros(1, 1, 128, 128))
    assert tuple(conv_out.shape[2:]) == grid
    assert m.pos_embed.shape[1] == grid[0] * grid[1] + m.cfg.num_prefix_tokens


@pytest.mark.parametrize("prefix,head_norm", [(1, False), (2, True)])
@pytest.mark.parametrize("batch", [1, 3])
def test_forward_shape(prefix, head_norm, batch):
    m = TinyAudioTransformer(TATConfig(num_prefix_tokens=prefix, head_norm=head_norm)).eval()
    assert m(torch.randn(batch, 128, 128)).shape == (batch, 35)


@pytest.mark.parametrize("shape", [(2, 63, 128), (2, 1, 128, 64), (2, 128, 64)])
def test_rejects_wrong_input_shape(shape):
    # The old pipeline produced 63 frames for a 1 s clip and crashed inside the pos_embed add
    with pytest.raises(ValueError, match="expected input"):
        TinyAudioTransformer()(torch.randn(*shape))


def test_eval_is_deterministic():
    m = TinyAudioTransformer().eval()
    x = torch.randn(2, 128, 128)
    torch.testing.assert_close(m(x), m(x))


def test_init_scale():
    m = TinyAudioTransformer()
    assert m.pos_embed.std().item() == pytest.approx(0.02, rel=0.2)
    assert m.prefix_tokens.abs().max().item() < 0.1  # trunc_normal_ bounds are absolute (±2), so this is ~4σ


@pytest.mark.parametrize("model", ["tat", "cnn"])
def test_overfits_one_batch(model):
    # A model that cannot memorise 32 examples has a bug (wiring, masking, gradient flow)
    torch.manual_seed(0)
    if model == "tat":
        m = TinyAudioTransformer(TATConfig(embed_dim=64, depth=2, num_heads=2, patch_stride=16, dropout=0.0))
    else:
        m = CNNBaseline(CNNConfig(dropout=0.0))  # 2-layer version is too small to memorise pure noise through avg-pool
    m.norm_in.set_stats(0.0, 1.0)
    x, y = torch.randn(32, 128, 128), torch.randint(0, 35, (32,))
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(300):
        m.train()
        loss = nn.functional.cross_entropy(m(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        assert (m(x).argmax(1) == y).all()

In [ ]:
%%writefile /kaggle/working/tat/tests/test_splits.py
from dataset import EXPECTED_SIZES, official_splits, speaker_id


def test_official_split_sizes(sc_root):
    splits, classes = official_splits(sc_root)
    assert len(classes) == 35
    assert {s: len(v) for s, v in splits.items()} == EXPECTED_SIZES


def test_speaker_disjoint(sc_root):
    splits, _ = official_splits(sc_root)
    speakers = {s: {speaker_id(r) for r, _ in v} for s, v in splits.items()}
    assert not speakers["train"] & speakers["val"]
    assert not speakers["train"] & speakers["test"]
    assert not speakers["val"] & speakers["test"]


def test_every_split_has_every_class(sc_root):
    # The old 80/10/10 slice of a label-sorted list put only the last label(s) in test
    splits, classes = official_splits(sc_root)
    for items in splits.values():
        assert {label for _, label in items} == set(range(len(classes)))


def test_no_background_noise_clips(sc_root):
    splits, _ = official_splits(sc_root)
    assert not any(r.startswith("_") for items in splits.values() for r, _ in items)

In [ ]:
%%writefile /kaggle/working/tat/tests/test_parity_ast.py
# Our architecture loaded with the authors' AST weights must compute the same function as their model.
import os
from pathlib import Path

import pytest
import torch

from conftest import hf_or_skip
from features import SAMPLE_RATE, fbank


@pytest.fixture(scope="module")
def models():
    return hf_or_skip()


def test_logits_match_on_random_input(models):
    ours, hf, fe = models
    g = torch.Generator().manual_seed(0)
    hf_in = torch.randn(2, fe.max_length, 128, generator=g)          # normalised, as HF expects
    raw = hf_in * (2 * fe.std) + fe.mean                             # what our model expects
    with torch.no_grad():
        torch.testing.assert_close(ours(raw), hf(hf_in).logits, atol=1e-3, rtol=1e-3)


def test_end_to_end_on_audio(models):
    # Waveform → our fbank → our model  vs  waveform → HF extractor → HF model
    ours, hf, fe = models
    root = os.environ.get("SC_ROOT")
    if root and Path(root, "testing_list.txt").exists():
        from dataset import load_wav
        rels = Path(root, "testing_list.txt").read_text().split()[::1000][:8]
        wavs = [load_wav(Path(root, r)) for r in rels]
        labels = [r.split("/")[0] for r in rels]
    else:
        g = torch.Generator().manual_seed(0)
        wavs, labels = [0.1 * torch.randn(SAMPLE_RATE, generator=g) for _ in range(2)], None

    with torch.no_grad():
        ours_logits = torch.stack([ours(fbank(w, 128, fe.max_length)[None])[0] for w in wavs])
        hf_logits = hf(**fe([w.numpy() for w in wavs], sampling_rate=SAMPLE_RATE, return_tensors="pt")).logits
    torch.testing.assert_close(ours_logits, hf_logits, atol=1e-3, rtol=1e-3)

    if labels:  # the reference itself should be right on real test clips, or we are comparing to garbage
        id2label = hf.config.id2label
        preds = [id2label[i] for i in ours_logits.argmax(1).tolist()]
        assert sum(p == l for p, l in zip(preds, labels)) >= len(labels) - 1, list(zip(preds, labels))

## Phase 0: data, official split, feature cache
First run takes roughly 5–10 min (download, extraction, then fbank for ~106k clips on 4 CPU cores). Re-runs reuse `/tmp`.

In [ ]:
DATA, CACHE = "/tmp/sc_v2", "/tmp/sc_cache"
!python prepare_data.py --root {DATA} --cache-dir {CACHE} --num-workers 4

## Phase 1: tests
Every test should **pass**, and none should be skipped. Skips mean the dataset or the Hub weren't reachable.
- `test_splits`: official sizes, speaker-disjoint, every class in every split
- `test_features`: our fbank matches torchaudio's Kaldi fbank and the HF AST feature extractor
- `test_model`: patch arithmetic, shape errors, init, overfitting a batch
- `test_parity_ast`: our architecture with the authors' weights matches the authors' model, on real test clips

In [ ]:
!SC_ROOT={DATA} python -m pytest tests -v -rs -p no:cacheprovider

## Learnability: memorise one real batch
Both must print `PASS`. If a model can't memorise 128 clips, it has a bug no hyperparameter will fix.

In [ ]:
!python train.py --model tat --cache-dir {CACHE} --overfit-batch 128
!python train.py --model cnn --cache-dir {CACHE} --overfit-batch 128

## Sanity training run: TinyAudioTransformer on GPU 0, CNN baseline on GPU 1
No augmentation yet (Phase 2). The checkpoint is chosen on val, and test is evaluated once at the end. Outputs go to `/kaggle/working/runs/<name>/`: `best.pt`, `results.json`, `history.json`, `test_logits.pt`, `train.log`.

In [ ]:
EPOCHS = 30
RUNS = {
    "tat_s0": ["--model", "tat", "--seed", "0"],
    "cnn_s0": ["--model", "cnn", "--seed", "0"],
}
n_gpu = max(1, torch.cuda.device_count())
procs = {}
for i, (name, extra) in enumerate(RUNS.items()):
    out = pathlib.Path(f"/kaggle/working/runs/{name}"); out.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(i % n_gpu)}
    cmd = [sys.executable, "train.py", "--cache-dir", CACHE, "--out-dir", str(out), "--epochs", str(EPOCHS), *extra]
    procs[name] = subprocess.Popen(cmd, stdout=open(out / "train.log", "w"), stderr=subprocess.STDOUT, env=env)

def last_line(name):
    lines = pathlib.Path(f"/kaggle/working/runs/{name}/train.log").read_text().strip().splitlines()
    return lines[-1] if lines else "starting…"

while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    for name in procs:
        print(f"{name}: {last_line(name)}")
    print("-" * 60, flush=True)

for name, p in procs.items():
    print(f"== {name} (exit {p.returncode})")
    print(pathlib.Path(f"/kaggle/working/runs/{name}/train.log").read_text()[-1500:])

## Results

In [ ]:
for name in RUNS:
    r = json.load(open(f"/kaggle/working/runs/{name}/results.json"))
    worst = sorted(r["per_class_test_acc"].items(), key=lambda kv: kv[1])[:5]
    print(f"{name:7s} {r['params'] / 1e6:5.2f}M params | best val {r['best_val_acc']:.2f}% (epoch {r['best_epoch']}) "
          f"| TEST {r['test_acc']:.2f}% | {r['train_time_s'] / 60:.1f} min")
    print("        weakest classes:", ", ".join(f"{c} {a:.1f}%" for c, a in worst))
print("\nReference: AST, 87M params, ImageNet-pretrained, 35-word SC v2 = 98.11% (paper). Not a target for a 0.85M scratch model.")
print("Phase 1 bar: the transformer should beat the CNN baseline on identical features.")

## Send back
1. The pytest summary line (and any failure output).
2. The two `PASS`/`FAIL` overfit lines.
3. The output of the Results cell.

Checkpoints are saved to `/kaggle/working/runs/`. Save a version of the notebook to keep them.